In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

import cooler

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.stats import pearsonr

from matplotlib_venn import venn3
from sklearn.metrics import adjusted_rand_score

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/pseudobulk_hic'

### consolidate shared compartments

In [ ]:
igvf_comps_path = f'{projdir}/csv/compartments/cooltools/ips_cluster/merged.comps.100kb.csv'
igvf_comps = pd.read_csv(igvf_comps_path, sep='\t')
igvf_comps.index = [f'{x}_{y}' for x,y in zip(igvf_comps['chrom'], igvf_comps['start'])]
igvf_comps.shape

In [ ]:
alt_comps = igvf_comps[igvf_comps['sample'] == 'IPS_alt']
main_comps = igvf_comps[igvf_comps['sample'] == 'IPS_main']
alt_comps.shape, main_comps.shape

In [ ]:
s = 'ipsc_rues2'
ref_comps_path = f'/u/project/cluo/terencew/public_datasets/4dn/compartments/cooltools/{s}.100kb.csv'
ref_comps = pd.read_csv(ref_comps_path, sep='\t')
ref_comps.index = [f'{x}_{y}' for x,y in zip(ref_comps['chrom'], ref_comps['start'])]

ref_comps.shape

In [ ]:
comps_set = np.concatenate([alt_comps.index, main_comps.index, ref_comps.index])
comps_set.shape

In [ ]:
tmp_comps = np.unique(comps_set, return_counts=True)[0]
counts = np.unique(comps_set, return_counts=True)[1]
mask = counts == 3
final_comps = tmp_comps[mask]
tmp = alt_comps.reindex(final_comps)
tmp['chr_num'] = [int(x.split('chr')[1]) for x in tmp['chrom']]
final_comps = tmp.sort_values(by=['chr_num', 'start']).index
final_comps.shape

In [ ]:
final_alt = alt_comps.reindex(final_comps)
final_main = main_comps.reindex(final_comps)
final_ref = ref_comps.reindex(final_comps)
final_alt.shape, final_main.shape, final_ref.shape

In [ ]:
eigs_df = pd.DataFrame(zip(final_alt['E1'], final_main['E1'], final_ref['E1']), index=final_alt.index)
eigs_df.columns = ['IPS_alt', 'IPS_main', '4DN']
eigs_df.head()

In [ ]:
a = eigs_df['IPS_alt']
b = eigs_df['4DN']
alt_mature = pearsonr(a, b)[0]
alt_mature

In [ ]:
a = eigs_df['IPS_main']
b = eigs_df['4DN']
main_mature = pearsonr(a, b)[0]
main_mature

In [ ]:
to_plot = pd.DataFrame()
to_plot['pearson_r'] = [alt_mature, main_mature]
to_plot['ips_subtype'] = ['IPS_alt', 'IPS_main']
sns.barplot(to_plot, x='ips_subtype', y='pearson_r', hue='ips_subtype')

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5/'
to_plot.to_csv(f'{outdir}/4dn_mature_ips_eigenvector_corr.csv', sep='\t')

### check actual compartment identity

In [ ]:
comps_df = pd.DataFrame(zip(final_alt['E1_comp'], final_main['E1_comp'], final_ref['E1_comp']), index=final_alt.index)
comps_df.columns = ['IPS_alt', 'IPS_main', '4DN']
comps_df.head()

In [ ]:
a_comps = comps_df[comps_df['4DN'] == 'A']
b_comps = comps_df[comps_df['4DN'] == 'B']

In [ ]:
(comps_df['IPS_alt'] == comps_df['4DN']).mean(), \
(comps_df['IPS_main'] == comps_df['4DN']).mean()

In [ ]:
(a_comps['IPS_alt'] == a_comps['4DN']).mean(), \
(a_comps['IPS_main'] == a_comps['4DN']).mean()

In [ ]:
(b_comps['IPS_alt'] == b_comps['4DN']).mean(), \
(b_comps['IPS_main'] == b_comps['4DN']).mean()

### check eigs again

In [ ]:
a_eigs = eigs_df.reindex(a_comps.index)
b_eigs = eigs_df.reindex(b_comps.index)

In [ ]:
a = a_eigs['IPS_alt']
b = a_eigs['4DN']
a_r1 = pearsonr(a, b)[0]

a = a_eigs['IPS_main']
a_r2 = pearsonr(a, b)[0]
a_r1, a_r2

In [ ]:
a = b_eigs['IPS_alt']
b = b_eigs['4DN']
b_r1 = pearsonr(a, b)[0]

a = b_eigs['IPS_main']
b_r2 = pearsonr(a, b)[0]
b_r1, b_r2

In [ ]:
corr_df = pd.DataFrame([[a_r1, a_r2],
              [b_r1, b_r2]])
corr_df.columns = ['IPS_alt', 'IPS_main']
corr_df['comp'] =  ['A', 'B']
to_plot = corr_df.melt(id_vars=['comp'], value_vars=['IPS_alt', 'IPS_main'])
to_plot

In [ ]:
sns.barplot(to_plot, x='comp', y='value', hue='variable')

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5/'
to_plot.to_csv(f'{outdir}/4dn_mature_ips_eigenvector_corr.csv', sep='\t')

### venn diagram

In [ ]:
alt_A  = set(comps_df.index[comps_df["IPS_alt"]  == "A"])
main_A = set(comps_df.index[comps_df["IPS_main"] == "A"])
dn_A   = set(comps_df.index[comps_df["4DN"]      == "A"])
venn3([alt_A, main_A, dn_A])

In [ ]:
alt_B  = set(comps_df.index[comps_df["IPS_alt"]  == "B"])
main_B = set(comps_df.index[comps_df["IPS_main"] == "B"])
dn_B   = set(comps_df.index[comps_df["4DN"]      == "B"])
venn3([alt_B, main_B, dn_B])

In [ ]:
!date